# NB01 — Data Audit & Country Mapping

**Objective**: Inventory all 118 ZIP archives in `data/country/`.  
For each ZIP:
- Detect if it contains observed CSV data (`output_data_points.csv`)
- Identify the country using centroid reverse-geocoding (Natural Earth boundaries)
- Audit data completeness (properties present, station count, depth range, data sources)
- List raster layers available (GSOCmap, HoliSoils, OpenLandMap)

**Inputs**: `data/country/*` (ZIP archives)  
**Outputs**:
- `data/intermediate/audit/zip_inventory.csv` — one row per ZIP
- `data/intermediate/audit/property_coverage.csv` — one row per ZIP × property
- `data/intermediate/audit/raster_inventory.csv` — one row per ZIP × raster layer
- `logs/nb01_audit.log`

**STOP conditions**:
- Any ZIP that cannot be opened raises an error and is logged, not skipped silently
- If country detection fails for > 20% of ZIPs → STOP and report

In [12]:
import zipfile
import io
import os
import json
import logging
import csv as csv_module
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

# ── Paths ──────────────────────────────────────────────────────────────────
BASE        = Path('/home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive')
COUNTRY_DIR = BASE / 'data' / 'country'
AUDIT_DIR   = BASE / 'data' / 'intermediate' / 'audit'
LOG_DIR     = BASE / 'logs'
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ── Logging ────────────────────────────────────────────────────────────────
log_path = LOG_DIR / 'nb01_audit.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(log_path, mode='w'),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)
log.info(f'NB01 started at {datetime.now().isoformat()}')

2026-03-19 16:00:23,315 [INFO] NB01 started at 2026-03-19T16:00:23.315279


In [13]:
# ── Load Natural Earth country boundaries for reverse-geocoding ─────────────
# naturalearth_lowres was removed from geopandas >= 1.0.
# We locate the copy bundled with pyogrio (already on disk in the active conda env).
import glob as _glob, sys

def _find_naturalearth_shp() -> str:
    """Locate naturalearth_lowres.shp in the active conda environment."""
    env_prefix = sys.prefix
    candidates = _glob.glob(
        f'{env_prefix}/**/naturalearth_lowres/naturalearth_lowres.shp', recursive=True
    )
    if candidates:
        return candidates[0]
    # Hard fallback: search all known conda envs
    candidates = _glob.glob(
        '/home/agbelgaid/anaconda3/**/naturalearth_lowres.shp', recursive=True
    )
    if candidates:
        return candidates[0]
    raise FileNotFoundError(
        'naturalearth_lowres.shp not found. '
        'Install geopandas via: conda install -c conda-forge geopandas'
    )

NE_SHP = _find_naturalearth_shp()
log.info(f'Using Natural Earth shapefile: {NE_SHP}')

world = gpd.read_file(NE_SHP)[['name', 'iso_a3', 'geometry']].copy()
world = world[world.geometry.notna()].reset_index(drop=True)

# Fix known iso_a3 = '-99' (geopandas lowres quirk for certain territories)
_ISO_FIXES = {
    'France':     'FRA',
    'Norway':     'NOR',
    'N. Cyprus':  'CYP',
    'Kosovo':     'XKX',
    'Somaliland': 'SOM',
}
mask = world['iso_a3'] == '-99'
world.loc[mask, 'iso_a3'] = world.loc[mask, 'name'].map(_ISO_FIXES)
log.info(f'Loaded {len(world)} country polygons (fixed {mask.sum()} iso_a3=-99 entries)')

def reverse_geocode_centroid(lats: list, lons: list) -> dict:
    """Return country name and ISO3 for the centroid of a list of coordinates."""
    if not lats:
        return {'country': None, 'iso3': None, 'method': 'no_coords'}

    clat = float(np.median(lats))
    clon = float(np.median(lons))
    point = Point(clon, clat)

    # Exact containment
    match = world[world.geometry.contains(point)]
    if len(match) > 0:
        row = match.iloc[0]
        return {'country': row['name'], 'iso3': row['iso_a3'], 'method': 'centroid_exact'}

    # Nearest polygon fallback (coastal / border stations)
    # Use .copy() to avoid SettingWithCopyWarning
    _w = world.copy()
    _w['dist'] = _w.geometry.distance(point)
    nearest = _w.loc[_w['dist'].idxmin()]
    return {'country': nearest['name'], 'iso3': nearest['iso_a3'], 'method': 'centroid_nearest'}

2026-03-19 16:00:30,650 [INFO] Using Natural Earth shapefile: /home/agbelgaid/anaconda3/envs/agent/lib/python3.1/site-packages/pyogrio/tests/fixtures/naturalearth_lowres/naturalearth_lowres.shp
2026-03-19 16:00:30,665 [INFO] Loaded 177 country polygons (fixed 5 iso_a3=-99 entries)


In [14]:
# ── Scan all ZIP files ─────────────────────────────────────────────────────
PROPERTIES_EXPECTED = [
    'BD', 'Ca', 'CaCO3', 'CEC', 'CF', 'clay', 'EC', 'K',
    'Mg', 'N', 'Na', 'nematode', 'occ', 'P', 'pH', 'sand',
    'silt', 'TC', 'WR_gravimetric', 'WR_volumetric'
]

zip_files = sorted(COUNTRY_DIR.glob('*'))
zip_files = [f for f in zip_files if f.is_file()]
log.info(f'Found {len(zip_files)} files in {COUNTRY_DIR}')

inventory_rows = []       # one row per ZIP
coverage_rows  = []       # one row per ZIP × property
raster_rows    = []       # one row per ZIP × raster layer
errors         = []

for zip_path in zip_files:
    record = {
        'zip_hash': zip_path.name,
        'zip_size_mb': round(zip_path.stat().st_size / 1e6, 2),
        'has_csv': False,
        'n_rows_csv': 0,
        'n_stations': 0,
        'country': None,
        'iso3': None,
        'geocode_method': None,
        'lat_min': None, 'lat_max': None,
        'lon_min': None, 'lon_max': None,
        'data_sources': None,
        'depth_min_cm': None, 'depth_max_cm': None,
        'properties_present': None,
        'n_properties': 0,
        'has_GSOCmap': False,
        'has_HoliSoils': False,
        'has_OpenLandMap': False,
        'error': None
    }
    
    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            names = zf.namelist()
            
            # ── Raster inventory ──────────────────────────────────────────
            for n in names:
                if n.endswith('.tif'):
                    parts = n.split('/')
                    source_folder = parts[0] if len(parts) > 1 else 'root'
                    if source_folder == 'GSOCmap':     record['has_GSOCmap'] = True
                    if source_folder == 'HoliSoils':   record['has_HoliSoils'] = True
                    if source_folder == 'OpenLandMap': record['has_OpenLandMap'] = True
                    raster_rows.append({
                        'zip_hash': zip_path.name,
                        'source_folder': source_folder,
                        'filename': parts[-1],
                        'path_in_zip': n
                    })
            
            # ── CSV audit ────────────────────────────────────────────────
            if 'output_data_points.csv' in names:
                record['has_csv'] = True
                
                with zf.open('output_data_points.csv') as f:
                    df = pd.read_csv(
                        io.TextIOWrapper(f, encoding='utf-8'),
                        low_memory=False,
                        usecols=['lat', 'lon', 'property', 'upper_depth_cm',
                                 'lower_depth_cm', 'value', 'data_source']
                    )
                
                record['n_rows_csv'] = len(df)
                
                # Unique stations
                stations = df[['lat', 'lon']].drop_duplicates()
                record['n_stations'] = len(stations)
                
                # Coordinate range
                lats = df['lat'].dropna().tolist()
                lons = df['lon'].dropna().tolist()
                record['lat_min'] = round(min(lats), 4)
                record['lat_max'] = round(max(lats), 4)
                record['lon_min'] = round(min(lons), 4)
                record['lon_max'] = round(max(lons), 4)
                
                # Reverse geocode
                geo = reverse_geocode_centroid(lats, lons)
                record['country']       = geo['country']
                record['iso3']          = geo['iso3']
                record['geocode_method'] = geo['method']
                
                # Data sources
                record['data_sources'] = '|'.join(sorted(df['data_source'].dropna().unique()))
                
                # Depth range
                record['depth_min_cm'] = df['upper_depth_cm'].min()
                record['depth_max_cm'] = df['lower_depth_cm'].max()
                
                # Properties present
                props_present = sorted(df['property'].dropna().unique().tolist())
                record['properties_present'] = '|'.join(props_present)
                record['n_properties'] = len(props_present)
                
                # Per-property coverage (% non-null values for surface ≤10 cm)
                surf = df[df['upper_depth_cm'] <= 10].copy()
                n_surf_stations = surf[['lat', 'lon']].drop_duplicates().shape[0]
                
                for prop in PROPERTIES_EXPECTED:
                    prop_surf = surf[surf['property'] == prop]
                    n_obs = prop_surf['value'].notna().sum()
                    n_stations_with_prop = prop_surf[['lat', 'lon']].drop_duplicates().shape[0]
                    coverage_rows.append({
                        'zip_hash': zip_path.name,
                        'country': record['country'],
                        'iso3': record['iso3'],
                        'property': prop,
                        'n_surface_obs': n_obs,
                        'n_surface_stations': n_surf_stations,
                        'n_stations_with_prop': n_stations_with_prop,
                        'pct_stations_with_prop': round(
                            100 * n_stations_with_prop / n_surf_stations, 1
                        ) if n_surf_stations > 0 else 0
                    })
                
                log.info(f'  {zip_path.name} → {record["country"]} ({record["iso3"]}) | '
                         f'{record["n_stations"]} stations | '
                         f'{record["n_properties"]} properties')
            
            else:
                log.info(f'  {zip_path.name} → raster-only (no CSV)')
                # Still try to get country from hex_grid.geojson if present
                if 'hex_grid.geojson' in names:
                    try:
                        with zf.open('hex_grid.geojson') as hf:
                            hgeo = json.load(hf)
                        coords = []
                        for feat in hgeo.get('features', []):
                            geom = feat.get('geometry', {})
                            if geom.get('type') == 'Polygon':
                                for ring in geom['coordinates']:
                                    for c in ring:
                                        coords.append(c)
                        if coords:
                            lons_hex = [c[0] for c in coords]
                            lats_hex = [c[1] for c in coords]
                            geo = reverse_geocode_centroid(lats_hex, lons_hex)
                            record['country']       = geo['country']
                            record['iso3']          = geo['iso3']
                            record['geocode_method'] = geo['method'] + '_hex'
                    except Exception as e:
                        log.warning(f'    Could not parse hex_grid.geojson for {zip_path.name}: {e}')
    
    except zipfile.BadZipFile as e:
        msg = f'BadZipFile: {zip_path.name} — {e}'
        log.error(msg)
        record['error'] = str(e)
        errors.append(msg)
    except Exception as e:
        msg = f'Error processing {zip_path.name}: {e}'
        log.error(msg)
        record['error'] = str(e)
        errors.append(msg)
    
    inventory_rows.append(record)

log.info(f'Scan complete. {len(inventory_rows)} ZIPs processed, {len(errors)} errors.')

2026-03-19 16:00:43,301 [INFO] Found 118 files in /home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive/data/country
2026-03-19 16:00:43,354 [INFO]   223krk83 → Albania (ALB) | 83 stations | 15 properties
2026-03-19 16:00:43,389 [INFO]   224lf9p7 → Greece (GRC) | 338 stations | 18 properties
2026-03-19 16:00:43,433 [INFO]   224nbt59 → Greece (GRC) | 338 stations | 18 properties
2026-03-19 16:00:43,585 [INFO]   227yj7zt → France (FRA) | 3127 stations | 20 properties
2026-03-19 16:00:43,613 [INFO]   229at92m → Georgia (GEO) | 18 stations | 16 properties
2026-03-19 16:00:43,639 [INFO]   22h6n8tu → Papua New Guinea (PNG) | 16 stations | 13 properties
2026-03-19 16:00:43,641 [INFO]   22ofp4ol → raster-only (no CSV)
2026-03-19 16:00:43,730 [INFO]   22t5yo8e → Burkina Faso (BFA) | 2116 stations | 23 properties
2026-03-19 16:00:43,731 [INFO]   234vs9jl → raster-only (no CSV)
2026-03-19 16:00:43,767 [INFO]   23jo4hd3 → Burundi (BDI) | 29 stations | 17 properties
2026-03-19 16:00:43,768 [I

In [15]:
# ── Build DataFrames & Save ────────────────────────────────────────────────
df_inventory = pd.DataFrame(inventory_rows)
df_coverage  = pd.DataFrame(coverage_rows)
df_rasters   = pd.DataFrame(raster_rows)

# Save
df_inventory.to_csv(AUDIT_DIR / 'zip_inventory.csv', index=False)
df_coverage.to_csv(AUDIT_DIR / 'property_coverage.csv', index=False)
df_rasters.to_csv(AUDIT_DIR / 'raster_inventory.csv', index=False)
log.info(f'Saved: zip_inventory.csv ({len(df_inventory)} rows)')
log.info(f'Saved: property_coverage.csv ({len(df_coverage)} rows)')
log.info(f'Saved: raster_inventory.csv ({len(df_rasters)} rows)')

2026-03-19 16:01:38,630 [INFO] Saved: zip_inventory.csv (118 rows)
2026-03-19 16:01:38,630 [INFO] Saved: property_coverage.csv (1540 rows)
2026-03-19 16:01:38,631 [INFO] Saved: raster_inventory.csv (19016 rows)


In [16]:
# ── STOP CONDITIONS ────────────────────────────────────────────────────────
n_with_csv = df_inventory['has_csv'].sum()
n_no_country = df_inventory[df_inventory['has_csv'] & df_inventory['country'].isna()].shape[0]
pct_failed_geocode = 100 * n_no_country / n_with_csv if n_with_csv > 0 else 0

print('\n=== NB01 AUDIT SUMMARY ===')
print(f'Total ZIPs scanned       : {len(df_inventory)}')
print(f'ZIPs with observed CSV   : {n_with_csv}')
print(f'Raster-only ZIPs         : {len(df_inventory) - n_with_csv}')
print(f'Unique countries detected: {df_inventory["country"].nunique()}')
print(f'Geocoding failures       : {n_no_country} ({pct_failed_geocode:.1f}%)')
print(f'ZIP errors               : {len(errors)}')
print()

# Summarize by country
by_country = (
    df_inventory[df_inventory['has_csv']]
    .groupby(['country', 'iso3'])
    .agg(n_zips=('zip_hash', 'count'), total_stations=('n_stations', 'sum'))
    .sort_values('total_stations', ascending=False)
)
print('Top countries by station count:')
print(by_country.head(20).to_string())

# STOP if geocoding failure rate is too high
assert pct_failed_geocode <= 20, (
    f'STOP: {pct_failed_geocode:.1f}% of CSV ZIPs could not be geocoded '
    f'(threshold=20%). Check logs and fix manually.'
)

# STOP if critical errors
if errors:
    print(f'\n⚠ WARNING: {len(errors)} ZIP errors encountered (logged). '
          f'Investigate before proceeding to NB02.')
    for e in errors:
        print(f'  {e}')

log.info('NB01 completed successfully — proceed to NB02')

2026-03-19 16:01:46,610 [INFO] NB01 completed successfully — proceed to NB02



=== NB01 AUDIT SUMMARY ===
Total ZIPs scanned       : 118
ZIPs with observed CSV   : 77
Raster-only ZIPs         : 41
Unique countries detected: 26
Geocoding failures       : 0 (0.0%)
ZIP errors               : 12

Top countries by station count:
                           n_zips  total_stations
country              iso3                        
Australia            AUS        1           39727
Germany              DEU        5           21350
France               FRA        4           12508
Burkina Faso         BFA        3            6348
Botswana             BWA        4            5472
Angola               AGO        3            5400
Cameroon             CMR        3            4182
Netherlands          NLD        3            4062
Benin                BEN        4            2908
Greece               GRC        5            1690
Estonia              EST        5            1115
Albania              ALB        6             498
Croatia              HRV        6             474
Mo

In [ ]:
dup = by_country[by_country['n_zips'] > 1]
if len(dup) > 0:
    print('\nCountries with MULTIPLE ZIPs (verify these are separate regions, not duplicates):')
    print(dup.to_string())
    log.warning(f'{len(dup)} countries have multiple ZIP archives — manual review required')
else:
    print('\nNo duplicate country ZIPs detected.')

by_country.reset_index().to_csv(AUDIT_DIR / 'country_summary.csv', index=False)
log.info('Saved: country_summary.csv')

2026-03-19 16:02:39,204 [WARNING] 18 countries have multiple ZIP archives — manual review required
2026-03-19 16:02:39,208 [INFO] Saved: country_summary.csv



Countries with MULTIPLE ZIPs (verify these are separate regions, not duplicates):
                           n_zips  total_stations
country              iso3                        
Germany              DEU        5           21350
France               FRA        4           12508
Burkina Faso         BFA        3            6348
Botswana             BWA        4            5472
Angola               AGO        3            5400
Cameroon             CMR        3            4182
Netherlands          NLD        3            4062
Benin                BEN        4            2908
Greece               GRC        5            1690
Estonia              EST        5            1115
Albania              ALB        6             498
Croatia              HRV        6             474
Moldova              MDA        5             175
Central African Rep. CAF        2             166
Montenegro           MNE        5             125
Burundi              BDI        3              87
Algeria          